In [ ]:
import csv
import os

def copy_arabic_to_english_and_remove_ar_fields(input_filename="input.csv", output_filename="output.csv"):
    """
    Reads a CSV file, copies Arabic field values to corresponding English fields,
    removes the original Arabic fields, and writes the modified data to a new CSV file.
    Handles CSV files with a UTF-8 Byte Order Mark (BOM).

    Expected input CSV header structure includes fields like:
    question,option_a,...,answer,question_ar,option_a_ar,...,answer_ar
    """
    # Define the mapping from Arabic field names to English field names
    arabic_to_english_map = {
        'question_ar': 'question',
        'option_a_ar': 'option_a',
        'option_b_ar': 'option_b',
        'option_c_ar': 'option_c',
        'option_d_ar': 'option_d',
        'answer_ar': 'answer', # Note: Your CSV might have 'answer_a' instead.
    }

    try:
        # Use encoding 'utf-8-sig' to handle potential BOM at the start of the file
        with open(input_filename, mode='r', encoding='utf-8-sig', newline='') as infile, \
             open(output_filename, mode='w', encoding='utf-8', newline='') as outfile:

            reader = csv.DictReader(infile)

            # Validate input CSV header
            if not reader.fieldnames:
                print(f"Error: The CSV file '{input_filename}' is empty or has no header.")
                return

            input_header = list(reader.fieldnames) # Get the header from the reader
            print(f"Detected header: {', '.join(input_header)}") # Debugging: print detected header

            # Check if all necessary source Arabic fields are present in the input CSV header
            missing_source_fields = [field for field in arabic_to_english_map.keys() if field not in input_header]
            if missing_source_fields:
                print(f"Error: The input CSV file '{input_filename}' is missing one or more required Arabic source fields for mapping.")
                print(f"Missing source fields: {', '.join(missing_source_fields)}")
                print(f"Expected Arabic source fields: {', '.join(arabic_to_english_map.keys())}")
                print(f"Actual header: {', '.join(input_header)}")
                print("Please ensure your CSV header matches the expected Arabic field names (e.g., 'question_ar', 'answer_ar').")
                return

            # Check if all necessary target English fields are present in the input CSV header
            missing_target_fields = [field for field in arabic_to_english_map.values() if field not in input_header]
            if missing_target_fields:
                print(f"Error: The input CSV file '{input_filename}' is missing one or more required English target fields for mapping.")
                print(f"Missing target fields: {', '.join(missing_target_fields)}")
                print(f"Expected English target fields: {', '.join(set(arabic_to_english_map.values()))}")
                print(f"Actual header: {', '.join(input_header)}")
                print("Please ensure your CSV header matches the expected English field names (e.g., 'question', 'option_a').")
                return

            # Define the new header for the output file:
            # It will be all original headers MINUS the Arabic source fields
            arabic_source_fields_to_remove = set(arabic_to_english_map.keys())
            output_header = [field for field in input_header if field not in arabic_source_fields_to_remove]

            writer = csv.writer(outfile)
            writer.writerow(output_header) # Write the new, filtered header

            # Process each row
            for row_number, row_dict in enumerate(reader, 1):
                # Create a copy of the row to modify
                # Ensure all fields from input_header are present, even if empty in current row_dict
                updated_row_dict = {field: row_dict.get(field, "") for field in input_header}

                # Perform the copy operations (from Arabic to English fields)
                for arabic_field, english_field in arabic_to_english_map.items():
                    if arabic_field in updated_row_dict: # Source field exists
                        # Copy the value from the Arabic field to the English field.
                        # If updated_row_dict[arabic_field] is None or an empty string, it will copy that over.
                        updated_row_dict[english_field] = updated_row_dict[arabic_field]

                # Prepare the final output row based on the output_header
                # This ensures only the desired columns are written, in the correct order
                final_output_row_values = [updated_row_dict.get(field_name, "") for field_name in output_header]
                writer.writerow(final_output_row_values)

        print(f"Successfully processed the CSV file.")
        print(f"Copied Arabic fields to English fields, removed Arabic fields, and saved to '{output_filename}'")

    except FileNotFoundError:
        print(f"Error: The file '{input_filename}' was not found.")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        import traceback
        traceback.print_exc()


if __name__ == "__main__":
    print("CSV Arabic to English Field Copier & Cleaner")
    print("-------------------------------------------")

    input_file = input("Enter the path to your input CSV file (e.g., data.csv): ")

    if '.' in input_file:
        base, ext = os.path.splitext(input_file)
        suggested_output_file = f"{base}_processed{ext}"
    else:
        suggested_output_file = f"{input_file}_processed.csv"

    output_file = input(f"Enter the path for your output CSV file (press Enter for '{suggested_output_file}'): ")
    if not output_file:
        output_file = suggested_output_file

    copy_arabic_to_english_and_remove_ar_fields(input_file, output_file)

    # Example of how to create a dummy CSV for testing:
    # Ensure this sample matches the structure the script expects, especially all mapped fields.
    # with open("sample_input_v3_bom.csv", mode='w', encoding='utf-8-sig', newline='') as f: # Save with BOM for testing
    # # with open("sample_input_v3_nobom.csv", mode='w', encoding='utf-8', newline='') as f: # Save without BOM for testing
    #     writer = csv.writer(f)
    #     writer.writerow([
    #         "question","option_a","option_b","option_c","option_d","answer","extra_field_en",
    #         "question_ar","option_a_ar","option_b_ar","option_c_ar","option_d_ar","answer_ar","extra_field_ar"
    #         # If testing with 'answer_a', change "answer_ar" above to "answer_a"
    #     ])
    #     writer.writerow([
    #         "Old Q1","Old A1","Old B1","Old C1","Old D1","Old Ans1","Extra EN1",
    #         "سؤال ١ جديد","خيار أ ١ جديد","خيار ب ١ جديد","خيار ج ١ جديد","خيار د ١ جديد","إجابة ١ جديدة","Extra AR1"
    #     ])


CSV Arabic to English Field Copier & Cleaner
-------------------------------------------
Enter the path to your input CSV file (e.g., data.csv): parsed_questions_with_options_translated.csv
Enter the path for your output CSV file (press Enter for 'parsed_questions_with_options_translated_processed.csv'): 
Detected header: question, option_a, option_b, option_c, option_d, answer, question_ar, option_a_ar, option_b_ar, option_c_ar, option_d_ar, answer_ar
Successfully processed the CSV file.
Copied Arabic fields to English fields, removed Arabic fields, and saved to 'parsed_questions_with_options_translated_processed.csv'
